# Node 03 · 配体制备（master SDF → PDBQT）+ 正向原子映射  · v0.2.0
### 空间结构身份 → 对接就绪表示

调度 `core/node03_ligand_prep.py`（v0.2.0）。消费 Node 02 v0.2.0 的权威 3D 主分子，
为每个【已构建】分子产出对接就绪的 PDBQT + 显式带校验和的正向原子映射。

**三条封死 V2 缺陷**：Meeko（不重新质子化）；键级/立体/电荷不从 PDBQT 读回；映射按坐标恒等重建。

**v0.2.0 关键修订**：统一 `rigid_macrocycles=True` —— 禁用 Meeko 柔性大环拆分。
否则仅 28/1765 个分子会被拆环、获得额外构象自由度（并插入 G# 胶水赝原子），
使其在 BBS 分布中占不公平优势、污染设定 MD 阈值的聚类。环柔性在 MD 中充分采样。
同时映射器可识别并排除 G# 赝原子，QC 不再把 Meeko 特性误判为失败。

**QC 门**：每个真实 PDBQT 原子匹配到 master 原子（防加氢）；全部重原子+极性 H 被覆盖（防脱质子）；双射（防错配）。

**⚠ DRY_RUN** 默认 True。**内核**需 rdkit + meeko（如 ai_drug_design）。

In [4]:
# ── Cell 1 · 配置 ──────────────────────────────────────
from pathlib import Path
import sys
DRY_RUN = False                       # 核验无误后改 False 落盘
PROJECT_ROOT = Path("/mnt/hpc/home/wanghongmei/lang2").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
NODE02_DIR = PROJECT_ROOT / "results" / "02_master_molecules"
OUT_DIR    = PROJECT_ROOT / "results" / "03_ligand_pdbqt"
IDMAN_02   = NODE02_DIR / "identity_manifests.jsonl"
print("DRY_RUN    =", DRY_RUN)
print("NODE02_DIR =", NODE02_DIR, " idman 存在:", IDMAN_02.is_file())
print("OUT_DIR    =", OUT_DIR)

DRY_RUN    = False
NODE02_DIR = /mnt/hpc/home/wanghongmei/lang2/results/02_master_molecules  idman 存在: True
OUT_DIR    = /mnt/hpc/home/wanghongmei/lang2/results/03_ligand_pdbqt


In [5]:
# ── Cell 2 · 导入 + 读 Node 02 记录 + 链路核验 ──────────
import importlib, json
from core import node03_ligand_prep as N3
importlib.reload(N3)
from rdkit import Chem

print("node03 版本:", N3.NODE03_VERSION, "| rigid_macrocycles 默认:", N3.RIGID_MACROCYCLES)
print("meeko 可用:", N3.meeko_available(), "| 工具:", N3._tool_versions("meeko"))
assert N3.meeko_available(), "Meeko 未安装 —— 先 pip install meeko（Node 03 拒绝回退到 obabel-带-p）"

node02_recs = [json.loads(l) for l in IDMAN_02.read_text(encoding="utf-8").splitlines() if l.strip()]
N02_MANI = NODE02_DIR / "Node02_manifest.json"
fm = json.loads(N02_MANI.read_text(encoding="utf-8"))
assert len(node02_recs) == fm["stats"]["n_built"], (
    f"记录数 {len(node02_recs)} 与 Node02 manifest n_built {fm['stats']['n_built']} 不符")
print(f"\n读入 {len(node02_recs)} 条已构建记录")
print(f"Node 02 来源: v{fm['version']} | input_sha256 {fm['input_sha256'][:12]}…")
print(f"上游链: finalize v{fm['upstream_node01']['finalize_version']} "
      f"| 指认 {fm['upstream_node01']['input_sha256_declared'][:12]}…")

node03 版本: 0.2.0 | rigid_macrocycles 默认: True
meeko 可用: True | 工具: {'rdkit': '2025.09.4', 'meeko': '0.7.1'}

读入 1765 条已构建记录
Node 02 来源: v0.2.0 | input_sha256 14f1c83a26c0…
上游链: finalize v0.2.1 | 指认 b7abb39bf4da…


In [6]:
# ── Cell 3 · 冒烟测试（前 30 条）────────────────────────
SMOKE_N = 30
b, h, f, st = N3.run_node03(node02_recs[:SMOKE_N], Chem, OUT_DIR,
                            backend="meeko", dry_run=DRY_RUN)
print(f"冒烟 {SMOKE_N} 条: built={st['n_built']} held={st['n_held']} qc_fail={st['n_qc_fail']} "
      f"charge_warn={st['n_charge_warnings']} glue={st['n_with_glue_pseudo_atoms']}")
if st["qc_fail_reasons"]:
    print("  ⚠ QC 失败:", st["qc_fail_reasons"])
    for x in f[:5]: print(f"    {x['name']}: {x.get('reason')} — {str(x.get('error'))[:60]}")
else:
    print("  ✅ 无 QC 失败")

[node03] dry_run=False backend=meeko rigid_macrocycles=True -> 将写 PDBQT 到 /mnt/hpc/home/wanghongmei/lang2/results/03_ligand_pdbqt/pdbqt
冒烟 30 条: built=30 held=0 qc_fail=0 charge_warn=0 glue=0
  ✅ 无 QC 失败


In [7]:
# ── Cell 4 · 全量制备（约 1765 条）─────────────────────
RUN_FULL = True
if RUN_FULL:
    built, held, failures, stats = N3.run_node03(node02_recs, Chem, OUT_DIR,
                                                 backend="meeko", dry_run=DRY_RUN)
    print(f"全量: built={stats['n_built']} held={stats['n_held']} qc_fail={stats['n_qc_fail']}")
    print(f"  rigid_macrocycles={stats['rigid_macrocycles']} | 含胶水赝原子分子数={stats['n_with_glue_pseudo_atoms']} (应=0)")
    print(f"  held_reasons: {stats['held_reasons']}")
    print(f"  qc_fail_reasons: {stats['qc_fail_reasons']}")
    orph = stats["orphaned_parents"]
    print(f"  孤儿母体: {orph['n_orphaned_parents']}")
    if orph["n_orphaned_parents"]: print("   ⚠", orph["orphan_parent_ids"][:15])
    else: print("   ✅ 无孤儿母体")
else:
    print("[未运行全量] 设 RUN_FULL=True")

[node03] dry_run=False backend=meeko rigid_macrocycles=True -> 将写 PDBQT 到 /mnt/hpc/home/wanghongmei/lang2/results/03_ligand_pdbqt/pdbqt
全量: built=1765 held=0 qc_fail=0
  rigid_macrocycles=True | 含胶水赝原子分子数=0 (应=0)
  held_reasons: {}
  qc_fail_reasons: {}
  孤儿母体: 0
   ✅ 无孤儿母体


In [8]:
# ── Cell 5 · 写盘（仅 DRY_RUN=False）────────────────────
manifest, paths = N3.write_outputs(built, held, failures, stats,
                                   OUT_DIR, NODE02_DIR, backend="meeko", dry_run=DRY_RUN)
if not DRY_RUN:
    print("✓ 已写:")
    for k, p in paths.items(): print(f"  {k}: {p}")
    print(f"  PDBQT 目录: {OUT_DIR}/pdbqt/  | 原子映射: {OUT_DIR}/atom_maps/")
    print("\n✅ Node 03 完成。下一步：受体制备（6W63/3V3M/4YLU）+ Node 04 对接。")
else:
    print("[DRY_RUN] 未写盘。manifest 预览:\n")
    print(json.dumps(manifest, indent=2, ensure_ascii=False)[:1400])

✓ 已写:
  identity_manifests_node03: /mnt/hpc/home/wanghongmei/lang2/results/03_ligand_pdbqt/identity_manifests_node03.jsonl
  held: /mnt/hpc/home/wanghongmei/lang2/results/03_ligand_pdbqt/node03_held.csv
  failures: /mnt/hpc/home/wanghongmei/lang2/results/03_ligand_pdbqt/node03_qc_failures.json
  manifest: /mnt/hpc/home/wanghongmei/lang2/results/03_ligand_pdbqt/Node03_manifest.json
  PDBQT 目录: /mnt/hpc/home/wanghongmei/lang2/results/03_ligand_pdbqt/pdbqt/  | 原子映射: /mnt/hpc/home/wanghongmei/lang2/results/03_ligand_pdbqt/atom_maps/

✅ Node 03 完成。下一步：受体制备（6W63/3V3M/4YLU）+ Node 04 对接。


In [9]:
# ── Cell 6 · 落盘后独立复核（文件 hash + 映射 + 无重新质子化）──
from pathlib import Path
import json, hashlib
from rdkit import Chem
from core import node03_ligand_prep as N3

OUT_DIR = Path("/mnt/hpc/home/wanghongmei/lang2/results/03_ligand_pdbqt")
MASTER_SDF = Path("/mnt/hpc/home/wanghongmei/lang2/results/02_master_molecules/sdf")
recs = [json.loads(l) for l in (OUT_DIR / "identity_manifests_node03.jsonl")
        .read_text(encoding="utf-8").splitlines() if l.strip()]
print("records:", len(recs))

bad_pdbqt_hash, bad_map_hash, remap_mismatch, missing = [], [], [], []
for r in recs:
    pq = Path(r["pdbqt_path"]); mp = Path(r["atom_map_path"])
    if not pq.is_file() or not mp.is_file():
        missing.append(r["name"]); continue
    if hashlib.sha256(pq.read_bytes()).hexdigest() != r["pdbqt_sha256"]:
        bad_pdbqt_hash.append(r["name"])
    mo = json.loads(mp.read_text(encoding="utf-8"))
    recomputed = hashlib.sha256(json.dumps(mo["pdbqt_to_master"], sort_keys=True).encode()).hexdigest()
    if recomputed != r["map_checksum"] or mo["map_checksum"] != r["map_checksum"]:
        bad_map_hash.append(r["name"])
    sdf = MASTER_SDF / f'{r["name"]}.sdf'
    if not sdf.is_file(): continue
    mol = Chem.MolFromMolFile(str(sdf), sanitize=True, removeHs=False)
    pa = N3.parse_pdbqt_atoms(pq.read_text())
    fwd, unm, glue = N3.build_forward_map(mol, pa)          # v0.2.0: 三值解包（含 glue）
    stored = {int(k): v for k, v in mo["pdbqt_to_master"].items()}
    if unm or fwd != stored:
        remap_mismatch.append((r["name"], len(unm), len(glue), fwd == stored))

print(f"missing files      : {len(missing)}")
print(f"pdbqt hash drift   : {len(bad_pdbqt_hash)}")
print(f"map checksum drift : {len(bad_map_hash)}")
print(f"remap mismatch     : {len(remap_mismatch)}  (独立重建与存储映射不符 = 无声改动)")
for x in remap_mismatch[:10]: print("   ", x)
if not (missing or bad_pdbqt_hash or bad_map_hash or remap_mismatch):
    print("\n✅ 全部通过：文件未漂移、映射可独立复现、无重新质子化。Node 03 冻结。")

records: 1765
missing files      : 0
pdbqt hash drift   : 0
map checksum drift : 0
remap mismatch     : 0  (独立重建与存储映射不符 = 无声改动)

✅ 全部通过：文件未漂移、映射可独立复现、无重新质子化。Node 03 冻结。
